In [ ]:
"""
Match per-sample timepoints to an h5ad's obs['Timepoint'].

Assumes:
- A root directory containing sample folders like LSK_d2_1, LSK_d2_2,
  LSK_d4_1_2, LSK_d6_2_3, etc.
- Each sample folder contains three files: a matrix (*.mtx or *.mtx.gz),
  a genes file (genes.csv), and a barcodes file (barcodes.csv).
- The timepoint is encoded in the folder name as d2, d4, d6 (days 2/4/6).
- A separate .h5ad file whose .obs has a Timepoint column, and whose
  .obs_names are cell barcodes.

You will likely need to tweak:
  - TIMEPOINT_REGEX, if your folder naming differs from LSK_d{N}_...
  - the barcode-matching section, if h5ad barcodes don't look like barcodes.csv
"""
from twinfer.utils.paths import get_data_root as _twinfer_get_data_root
TWINFER_PROJECT_ROOT = _twinfer_get_data_root().parent  # [2026-09-30 added: replaces hardcoded project-root paths (/home/gzu5140/TwINFER_KA, /gpfs/projects/b1255/hzhang/TwINFER_KA, old Keerthana_b1042 tree)]

import re
import glob
import os

import numpy as np
import pandas as pd
import scipy.io as sio
import scipy.sparse as sp
import anndata as ad
import scanpy as sc
import subprocess


# ---- CONFIG: edit these ----

# Directory that contains the LSK_d2_1, LSK_d2_2, ... subfolders
SAMPLES_ROOT = f'{TWINFER_PROJECT_ROOT}/real_data/LARRY_new_raw/'

# Path to your h5ad file

cmd = """
module load git-lfs
git lfs pull --include=real_data/LSK_d2_d4_d6.h5ad
"""



result = subprocess.run(
    ["bash", "-lc", cmd],
    capture_output=True,
    text=True,
)

In [ ]:
from twinfer.utils.paths import get_data_root as _twinfer_get_data_root
TWINFER_PROJECT_ROOT = _twinfer_get_data_root().parent  # [2026-09-30 added: replaces hardcoded project-root paths (/home/gzu5140/TwINFER_KA, /gpfs/projects/b1255/hzhang/TwINFER_KA, old Keerthana_b1042 tree)]
H5AD_PATH = f'{TWINFER_PROJECT_ROOT}/clean_data/real_data/LSK_d2_d4_d6.h5ad'
H5AD_LIBRARY_COL = "Library"

# Column in h5ad.obs holding the timepoint
H5AD_TIMEPOINT_COL = 'Time point'

# Regex to pull the timepoint number out of a folder name like "LSK_d4_1_2" -> 4
TIMEPOINT_REGEX = re.compile(r"_d(\d+)_")

In [ ]:
# ---- Helpers ----
def parse_timepoint(folder_name: str) -> int:
    """Extract the integer timepoint (2, 4, 6, ...) from a sample folder name."""
    m = TIMEPOINT_REGEX.search(folder_name)
    if not m:
        raise ValueError(f"Could not parse timepoint from folder name: {folder_name!r}")
    return int(m.group(1))
 
 
def find_one(folder: str, patterns: list) -> str:
    """Find exactly one file in `folder` matching any of `patterns`, error otherwise."""
    hits = []
    for pat in patterns:
        hits.extend(glob.glob(os.path.join(folder, pat)))
    hits = sorted(set(hits))
    if len(hits) == 0:
        raise FileNotFoundError(f"No file matching {patterns} in {folder}")
    if len(hits) > 1:
        raise RuntimeError(f"Multiple files matching {patterns} in {folder}: {hits}")
    return hits[0]
 
 
def read_delimited(path: str) -> pd.DataFrame:
    """Read a barcodes/features file, detecting tab vs comma delimiter by
    peeking at the first line (more reliable here than pandas' sep=None
    sniffer, which can misfire when a field itself contains a comma).
    Transparently handles .gz via the file extension."""
    opener = gzip.open if path.endswith(".gz") else open
    with opener(path, "rt") as f:
        first_line = f.readline()
    delimiter = "\t" if first_line.count("\t") >= first_line.count(",") else ","
    return pd.read_csv(path, header=None, sep=delimiter, quoting=csv.QUOTE_NONE)
 
 
def read_sample_folder(folder: str) -> ad.AnnData:
    """Read one sample folder (matrix + features/genes + barcodes) into an AnnData."""
    mtx_path = find_one(folder, ["matrix.mtx", "matrix.mtx.gz", "*.mtx", "*.mtx.gz"])
    features_path = find_one(folder, [
        "features.tsv", "features.tsv.gz", "features.csv", "features.csv.gz",
        "genes.tsv", "genes.tsv.gz", "genes.csv", "genes.csv.gz",
    ])
    barcodes_path = find_one(folder, [
        "barcodes.tsv", "barcodes.tsv.gz", "barcodes.csv", "barcodes.csv.gz",
    ])
 
    mat = sio.mmread(mtx_path)
    mat = sp.csr_matrix(mat)
 
    features = read_delimited(features_path)
    barcodes = read_delimited(barcodes_path)
 
    # features.tsv is typically (ensembl_id, symbol, feature_type); use column 0 (ID)
    # unless you'd rather match on gene symbols (usually column 1) -- adjust below.
    gene_ids = features.iloc[:, 0].astype(str).values
    barcode_ids = barcodes.iloc[:, 0].astype(str).values
 
    n_obs_mat, n_var_mat = mat.shape
    # mtx is usually (genes x cells); transpose if it matches barcodes on the wrong axis
    if n_obs_mat == len(gene_ids) and n_var_mat == len(barcode_ids):
        mat = mat.T
    elif n_obs_mat == len(barcode_ids) and n_var_mat == len(gene_ids):
        pass
    else:
        raise ValueError(
            f"Matrix shape {mat.shape} in {folder} doesn't match "
            f"{len(gene_ids)} genes / {len(barcode_ids)} barcodes"
        )
 
    adata = ad.AnnData(X=mat)
    adata.var_names = gene_ids
    adata.obs_names = barcode_ids
    return adata
 
 
def strip_suffix(bc: str) -> str:
    """Remove all hyphens from a barcode for comparison purposes."""
    return bc.replace("-", "")

In [ ]:
import gzip
import csv
# ---- Read every sample folder ----
sample_dirs = sorted(
        d for d in glob.glob(os.path.join(SAMPLES_ROOT, "LSK_d*"))
        if os.path.isdir(d)
)
print(f"found {len(sample_dirs)} sample folders")

adatas = []
sample_summary = []

for d in sample_dirs:
    sample_name = os.path.basename(d)
    tp = parse_timepoint(sample_name)
    a = read_sample_folder(d)
    a.obs["sample"] = sample_name
    a.obs["folder_timepoint"] = tp
    adatas.append(a)
    sample_summary.append({"sample": sample_name, "timepoint": tp, "n_cells": a.n_obs})

summary_df = pd.DataFrame(sample_summary)
print(summary_df)

In [ ]:
# ---- Load the h5ad ----
h5ad = sc.read_h5ad(H5AD_PATH)
print(h5ad)
print("h5ad Timepoint value counts:")
print(h5ad.obs[H5AD_TIMEPOINT_COL].value_counts())
 
combined = ad.concat(adatas, join="outer", index_unique=None)
# ---- Sanity check: do the barcode strings actually look alike? ----
print("sample of barcodes from folders:", list(combined.obs_names[:5]))
print("sample of h5ad obs_names:", list(h5ad.obs_names[:5]))
 
# ---- Match cells between the two datasets by (sample/library, barcode) ----
    # Barcode alone is NOT enough: 10x barcodes are drawn from a shared whitelist,
    # so the same barcode string legitimately recurs across different samples/libraries.
    # h5ad's Library column values match our folder names exactly, so combine that
    # with the (hyphen-stripped) barcode to get a key that's actually unique per cell.

combined.obs["_stripped_bc"] = [strip_suffix(b) for b in combined.obs_names]
h5ad.obs["_stripped_bc"] = [strip_suffix(b) for b in h5ad.obs_names]
 
combined.obs["_match_key"] = combined.obs["sample"].astype(str) + "_" + combined.obs["_stripped_bc"]
h5ad.obs["_match_key"] = h5ad.obs[H5AD_LIBRARY_COL].astype(str) + "_" + h5ad.obs["_stripped_bc"]
 
common_keys = pd.Index(combined.obs["_match_key"]).intersection(h5ad.obs["_match_key"])
print(f"matched cells: {len(common_keys)}")
 
    # ---- Build comparison table ----
folder_df = combined.obs[["_match_key", "sample", "folder_timepoint"]].copy()
h5ad_df = h5ad.obs[["_match_key", H5AD_TIMEPOINT_COL]].copy()
 
dup_folder = folder_df["_match_key"].duplicated().sum()
dup_h5ad = h5ad_df["_match_key"].duplicated().sum()
if dup_folder or dup_h5ad:
    print(f"WARNING: {dup_folder} duplicate keys in folder data, "
              f"{dup_h5ad} in h5ad -- keeping first occurrence "
              f"(these should now be rare true barcode collisions within one sample, "
              f"not cross-sample noise)")
    folder_df = folder_df.drop_duplicates("_match_key")
    h5ad_df = h5ad_df.drop_duplicates("_match_key")


In [ ]:
merged = folder_df.merge(h5ad_df, on="_match_key", how="inner")
print(f"{len(merged)} cells matched and compared")
 
merged["match"] = (
    merged["folder_timepoint"] == merged[H5AD_TIMEPOINT_COL]
)
 
print(f"SAME:      {merged['match'].sum()}")
print(f"DIFFERENT: {(~merged['match']).sum()}")
print("confusion matrix (rows = folder timepoint, cols = h5ad Timepoint):")
print(pd.crosstab(merged["folder_timepoint"], merged[H5AD_TIMEPOINT_COL]))

In [ ]:
# ---- Cells present in only one dataset ----
folder_keys = set(combined.obs["_match_key"])
h5ad_keys = set(h5ad.obs["_match_key"])

only_in_folders = folder_keys - h5ad_keys
only_in_h5ad = h5ad_keys - folder_keys
in_both = folder_keys & h5ad_keys

print(f"in loaded samples only: {len(only_in_folders)}")
print(f"in h5ad only:           {len(only_in_h5ad)}")
print(f"in both:                {len(in_both)}")

# breakdown of "only in loaded samples" by sample
only_folder_df = combined.obs[combined.obs["_match_key"].isin(only_in_folders)]
print()
print("cells only in loaded samples, by sample:")
print(only_folder_df.groupby("sample").size())

# breakdown of "only in h5ad" by Library
only_h5ad_df = h5ad.obs[h5ad.obs["_match_key"].isin(only_in_h5ad)]
print()
print(f"cells only in h5ad, by {H5AD_LIBRARY_COL}:")
print(only_h5ad_df.groupby(H5AD_LIBRARY_COL).size())

In [ ]:
from twinfer.utils.paths import get_data_root as _twinfer_get_data_root
TWINFER_PROJECT_ROOT = _twinfer_get_data_root().parent  # [2026-09-30 added: replaces hardcoded project-root paths (/home/gzu5140/TwINFER_KA, /gpfs/projects/b1255/hzhang/TwINFER_KA, old Keerthana_b1042 tree)]
from singletCode import check_sample_sheet, get_singlets

import pandas as pd
pathToInputSheet = f"{TWINFER_PROJECT_ROOT}/code/TwINFER/real_data/LARRY_barcodes_updatedSampleNames.csv"
df = pd.read_csv(pathToInputSheet, index_col=0)
df['sample'].unique()

In [ ]:
# ---- Collapse lineage barcodes via GRAPH CONNECTED COMPONENTS ----
# Matches the published LARRY methodology (Weinreb et al. 2020 and companion Klein-lab
# papers describe collapsing barcodes within a Hamming-distance threshold using a
# graph-connected-components algorithm) -- NOT the greedy first-match chaining used in
# the original notebook. This is deterministic / order-independent.
# Canonical representative per cluster = most frequent barcode.

import numpy as np
from scipy.spatial.distance import pdist, squareform

N_HAMMING = 3

class UnionFind:
    def __init__(self, n):
        self.parent = list(range(n))
        self.rank = [0] * n
    def find(self, x):
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]  # path compression
            x = self.parent[x]
        return x
    def union(self, x, y):
        rx, ry = self.find(x), self.find(y)
        if rx == ry:
            return
        if self.rank[rx] < self.rank[ry]:
            rx, ry = ry, rx
        self.parent[ry] = rx
        if self.rank[rx] == self.rank[ry]:
            self.rank[rx] += 1

unique_barcodes = df['barcode'].unique().tolist()
n = len(unique_barcodes)
print(f"unique raw barcodes: {n}")

bc_lengths = set(len(b) for b in unique_barcodes)
if len(bc_lengths) != 1:
    raise ValueError(f"barcodes have inconsistent lengths: {bc_lengths} -- Hamming distance needs equal length")
bc_len = bc_lengths.pop()

encoded = np.array([[ord(c) for c in bc] for bc in unique_barcodes], dtype=np.int32)
uf = UnionFind(n)

# Chunked pairwise comparison (avoids an n x n matrix in memory for large n)
CHUNK = 2000
n_edges = 0
for start in range(0, n, CHUNK):
    end = min(start + CHUNK, n)
    block = encoded[start:end]
    # compare this block against ALL barcodes with index > start (upper triangle only)
    compare_to = encoded[start:]  # only need indices >= start, since pairs with smaller
                                   # index were already handled when that earlier block ran
    diff_counts = (block[:, None, :] != compare_to[None, :, :]).sum(axis=2)
    rows_local, cols_local = np.where(diff_counts <= N_HAMMING)
    for r, c in zip(rows_local, cols_local):
        i_global = start + r
        j_global = start + c
        if i_global < j_global:
            uf.union(i_global, j_global)
            n_edges += 1
    if (start // CHUNK) % 5 == 0:
        print(f"processed {end} / {n} barcodes")

print(f"\nedges (pairs within Hamming <= {N_HAMMING}): {n_edges}")

# Group into connected components
components = {}
for idx, bc in enumerate(unique_barcodes):
    root = uf.find(idx)
    components.setdefault(root, []).append(bc)
print(f"connected components (final clusters): {len(components)}")

# Canonical = most frequent barcode within each component (freq = total row count)
barcode_totals = df['barcode'].value_counts()
bc_map = {}
for members in components.values():
    canonical = max(members, key=lambda b: barcode_totals.get(b, 0))
    for bc in members:
        bc_map[bc] = canonical

print(f"collapsed {n - len(components)} barcodes")

# Apply -- no dedup, no merging, one row = one UMI throughout
df_final = df.copy()
df_final['barcode'] = df_final['barcode'].map(bc_map)
df_final['nUMI'] = df_final.groupby(['cellID', 'sample', 'barcode'])['barcode'].transform('size')

assert len(df_final) == len(df), "row count changed unexpectedly"
print(f"\nfinal rows: {len(df_final)} == original rows: {len(df)}")
df_final.head()

In [ ]:
# ---- Diagnose chaining: flag components whose internal diameter exceeds N_HAMMING ----
# A component formed by a direct pairwise match would have max internal distance <= N_HAMMING.
# A larger diameter means some members are only connected via an intermediate bridge barcode
# (chaining), not because they're actually close to each other.
 
from itertools import combinations
def hamming(a, b):
    return sum(x != y for x, y in zip(a, b))
chained_components = []  # (members, diameter, bridging pairs over threshold)
max_diameter_overall = 0

for members in components.values():
    if len(members) < 2:
        continue
    pairwise = [(a, b, hamming(a, b)) for a, b in combinations(members, 2)]
    diameter = max(d for _, _, d in pairwise)
    max_diameter_overall = max(max_diameter_overall, diameter)

    if diameter > N_HAMMING:
        over_threshold_pairs = [(a, b, d) for a, b, d in pairwise if d > N_HAMMING]
        chained_components.append((members, diameter, over_threshold_pairs))

n_chained_barcodes = sum(len(members) for members, _, _ in chained_components)

print(f"components with diameter > N_HAMMING ({N_HAMMING}): {len(chained_components)}")
print(f"total barcodes involved in chained components: {n_chained_barcodes}")
print(f"max internal pairwise distance found (overall): {max_diameter_overall}")

# ---- Inspect chained components with frequency support, to judge whether merges look safe ----

print(f"{len(chained_components)} chained components, sorted by diameter:\n")

for members, diameter, over_pairs in sorted(chained_components, key=lambda x: -x[1]):
    freqs = {bc: barcode_totals.get(bc, 0) for bc in members}
    sorted_members = sorted(freqs.items(), key=lambda x: -x[1])

    top_freq = sorted_members[0][1]
    second_freq = sorted_members[1][1] if len(sorted_members) > 1 else 0
    # ratio of the 2nd-most-supported barcode to the top one -- low ratio = probably safe merge,
    # high ratio (closer to 1) = two comparably-supported barcodes got merged, worth reviewing
    ratio = second_freq / top_freq if top_freq > 0 else float('nan')

    flag = "  <-- REVIEW: comparable support, possible false merge" if ratio > 0.2 else ""
    print(f"diameter={diameter}, size={len(members)}, top2_ratio={ratio:.3f}{flag}")
    for bc, freq in sorted_members:
        print(f"    {bc}  (support: {freq})")
    print()

In [ ]:
# ---- Fix the flagged chaining cases with complete-linkage re-clustering ----
# Complete-linkage requires ALL pairwise distances within a sub-cluster to be <= N_HAMMING,
# so it cannot chain through a weak bridging barcode the way connected-components can.

from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform

# 1. Identify the 3 flagged components to fix (top2_ratio > 0.2)
components_to_fix = []
for members, diameter, over_pairs in chained_components:
    freqs = sorted(((bc, barcode_totals.get(bc, 0)) for bc in members), key=lambda x: -x[1])
    ratio = (freqs[1][1] / freqs[0][1]) if len(freqs) > 1 and freqs[0][1] > 0 else 0
    if ratio > 0.2:
        components_to_fix.append(members)

print(f"re-clustering {len(components_to_fix)} flagged components")

# 2. Re-cluster each with complete-linkage, cut at N_HAMMING
for members in components_to_fix:
    m = len(members)
    dist_matrix = np.zeros((m, m))
    for i in range(m):
        for j in range(i + 1, m):
            d = hamming(members[i], members[j])
            dist_matrix[i, j] = dist_matrix[j, i] = d

    condensed = squareform(dist_matrix)
    Z = linkage(condensed, method='complete')
    sub_labels = fcluster(Z, t=N_HAMMING, criterion='distance')

    # group members by their new sub-cluster label
    sub_clusters = {}
    for idx, label in enumerate(sub_labels):
        sub_clusters.setdefault(label, []).append(members[idx])

    print(f"\noriginal component ({m} members) split into {len(sub_clusters)} sub-clusters:")
    for label, sub_members in sub_clusters.items():
        # new canonical = most frequent barcode within this sub-cluster
        canonical = max(sub_members, key=lambda b: barcode_totals.get(b, 0))
        print(f"  sub-cluster canonical={canonical}: {sub_members}")
        # overwrite bc_map for these barcodes with the corrected, stricter assignment
        for bc in sub_members:
            bc_map[bc] = canonical

# 3. Re-apply the corrected bc_map to df
df_final['barcode'] = df['barcode'].map(bc_map)
df_final['nUMI'] = df_final.groupby(['cellID', 'sample', 'barcode'])['barcode'].transform('size')

assert len(df_final) == len(df), "row count changed unexpectedly"
print(f"\nfinal unique canonical barcodes after fix: {len(set(bc_map.values()))}")

In [ ]:
df_final = df_final[['cellID', 'barcode', 'sample']]
df_final = df_final[df_final['sample'].str.startswith("LSK")]

In [ ]:
from singletCode import check_sample_sheet, get_singlets

cellLabelList, stats = get_singlets(df_final)

In [ ]:
# ---- Apply the two published filtering steps before clone calling ----
# (i) drop (Cell-BC, Lineage-BC) pairs supported by <3 UMIs
# (ii) drop (Cell-BC, Lineage-BC) pairs that appear identically in >1 sequencing library (sample)
#      -- these can only arise from droplet-emulsion cross-contamination between libraries

MIN_UMIS = 3

# Start from df_final, using the corrected canonical barcode (barcode_merged)
pairs = df_final.groupby(['cellID', 'sample', 'barcode']).size().reset_index(name='nUMI')
print(f"total (cellID, sample, barcode) pairs before filtering: {len(pairs)}")

# ---- Step (i): UMI floor ----
pairs_umi_filtered = pairs[pairs['nUMI'] >= MIN_UMIS].copy()
print(f"after UMI floor (>= {MIN_UMIS}): {len(pairs_umi_filtered)} pairs "
      f"({len(pairs) - len(pairs_umi_filtered)} dropped)")

# ---- Step (ii): cross-library collision filter ----
# Note: "Cell-BC" here is the raw cellID WITHOUT the sample prefix -- since the same
# 10x barcode legitimately recurs across libraries, the paper's point is that seeing
# the exact same (cellID, lineage-barcode) combo in two DIFFERENT samples is only
# plausible via emulsion contamination (given how large the lineage-barcode space is).
pair_key = pairs_umi_filtered['cellID'].astype(str) + "||" + pairs_umi_filtered['barcode'].astype(str)
n_libraries_per_pair = pairs_umi_filtered.groupby(pair_key)['sample'].nunique()
colliding_pairs = set(n_libraries_per_pair[n_libraries_per_pair > 1].index)

is_colliding = pair_key.isin(colliding_pairs)
print(f"\n(Cell-BC, Lineage-BC) pairs appearing in >1 library: {len(colliding_pairs)} distinct pairs, "
      f"{is_colliding.sum()} rows discarded")

pairs_final = pairs_umi_filtered[~is_colliding].copy()
print(f"\nfinal filtered pairs: {len(pairs_final)} "
      f"(started from {len(pairs)} total)")

pairs_final.head()

In [ ]:
# ---- Rerun dominant vs exact-match comparison using the FILTERED pairs ----

# ---------------------------------------------------------------
# Method 1: DOMINANT barcode clone (on filtered pairs)
# ---------------------------------------------------------------
dominant_barcode_filtered = (
    pairs_final.sort_values('nUMI', ascending=False)
    .assign(_match_key=lambda d: d['sample'].astype(str) + "_" + d['cellID'].astype(str))
    .drop_duplicates('_match_key')
    .set_index('_match_key')['barcode']
)

clone_sizes_dominant_f = dominant_barcode_filtered.value_counts()
print(f"[Dominant, filtered] number of clones: {len(clone_sizes_dominant_f)}")
print(f"[Dominant, filtered] clone size distribution:\n{clone_sizes_dominant_f.describe()}")

# ---------------------------------------------------------------
# Method 2: ALL barcodes must match exactly (on filtered pairs)
# ---------------------------------------------------------------
pairs_final['_match_key'] = pairs_final['sample'].astype(str) + "_" + pairs_final['cellID'].astype(str)
barcode_sets_f = pairs_final.groupby('_match_key')['barcode'].apply(lambda bcs: frozenset(bcs.unique()))

clone_sizes_allbc_f = barcode_sets_f.value_counts()
print(f"\n[All barcodes, exact match, filtered] number of clones: {len(clone_sizes_allbc_f)}")
print(f"[All barcodes, exact match, filtered] clone size distribution:\n{clone_sizes_allbc_f.describe()}")

cell_to_allbc_clone_f = barcode_sets_f.apply(lambda s: ','.join(sorted(s)))

# ---------------------------------------------------------------
# Compare
# ---------------------------------------------------------------
compare_df_f = pd.DataFrame({
    'cell': dominant_barcode_filtered.index,
    'dominant_clone': dominant_barcode_filtered.values,
})
compare_df_f['allbc_clone'] = compare_df_f['cell'].map(cell_to_allbc_clone_f)
n_barcodes_f = pairs_final.groupby('_match_key')['barcode'].nunique()
compare_df_f['n_barcodes'] = compare_df_f['cell'].map(n_barcodes_f)

single_bc_f = (compare_df_f['n_barcodes'] == 1).sum()
multi_bc_f = (compare_df_f['n_barcodes'] > 1).sum()
print(f"\ncells with exactly 1 barcode (post-filter): {single_bc_f}")
print(f"cells with >1 barcode (post-filter): {multi_bc_f}")

split_check_f = compare_df_f.groupby('dominant_clone')['allbc_clone'].nunique()
split_dominant_f = split_check_f[split_check_f > 1]
print(f"\ndominant-barcode clones that get SPLIT (post-filter): {len(split_dominant_f)}")

# ---------------------------------------------------------------
# Direct before/after summary
# ---------------------------------------------------------------
print("\n=== BEFORE vs AFTER filtering ===")
print(f"{'Metric':<45}{'Before':>12}{'After':>12}")
print(f"{'Dominant clones':<45}{5792:>12}{len(clone_sizes_dominant_f):>12}")
print(f"{'Exact-match clones':<45}{8942:>12}{len(clone_sizes_allbc_f):>12}")
print(f"{'Dominant clones that get split':<45}{1860:>12}{len(split_dominant_f):>12}")

In [ ]:
loaded_sample_names = set(combined.obs["sample"].unique())

# ===================================================================
# STEP A: Restrict to the 15 loaded samples (creates pairs_final_restricted, compare_df_r)
# ===================================================================
pairs_final_restricted = pairs_final[pairs_final['sample'].isin(loaded_sample_names)].copy()
print(f"pairs_final: {len(pairs_final)} rows across {pairs_final['sample'].nunique()} samples")
print(f"pairs_final_restricted: {len(pairs_final_restricted)} rows across "
      f"{pairs_final_restricted['sample'].nunique()} samples (should be 15)")

dominant_barcode_r = (
    pairs_final_restricted.sort_values('nUMI', ascending=False)
    .assign(_match_key=lambda d: d['sample'].astype(str) + "_" + d['cellID'].astype(str))
    .drop_duplicates('_match_key')
    .set_index('_match_key')['barcode']
)
clone_sizes_dominant_r = dominant_barcode_r.value_counts()
print(f"\n[Dominant, filtered+restricted] number of clones: {len(clone_sizes_dominant_r)}")

pairs_final_restricted['_match_key'] = pairs_final_restricted['sample'].astype(str) + "_" + pairs_final_restricted['cellID'].astype(str)
barcode_sets_r = pairs_final_restricted.groupby('_match_key')['barcode'].apply(lambda bcs: frozenset(bcs.unique()))
clone_sizes_allbc_r = barcode_sets_r.value_counts()
print(f"[All barcodes, exact match, filtered+restricted] number of clones: {len(clone_sizes_allbc_r)}")

cell_to_allbc_clone_r = barcode_sets_r.apply(lambda s: ','.join(sorted(s)))
compare_df_r = pd.DataFrame({'cell': dominant_barcode_r.index, 'dominant_clone': dominant_barcode_r.values})
compare_df_r['allbc_clone'] = compare_df_r['cell'].map(cell_to_allbc_clone_r)
split_check_r = compare_df_r.groupby('dominant_clone')['allbc_clone'].nunique()
split_dominant_r = split_check_r[split_check_r > 1]
print(f"dominant-barcode clones that get SPLIT (filtered+restricted): {len(split_dominant_r)}")

print("\n=== BEFORE (unfiltered, 15 samples) vs AFTER (filtered, 15 samples) ===")
print(f"{'Metric':<45}{'Before':>12}{'After':>12}")
print(f"{'Dominant clones':<45}{5792:>12}{len(clone_sizes_dominant_r):>12}")
print(f"{'Exact-match clones':<45}{8942:>12}{len(clone_sizes_allbc_r):>12}")
print(f"{'Dominant clones that get split':<45}{1860:>12}{len(split_dominant_r):>12}")

# ===================================================================
# STEP B: Subset-compatible clone assignment
# ===================================================================
class UnionFind:
    def __init__(self, n):
        self.parent = list(range(n))
        self.rank = [0] * n
    def find(self, x):
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]
            x = self.parent[x]
        return x
    def union(self, x, y):
        rx, ry = self.find(x), self.find(y)
        if rx == ry:
            return
        if self.rank[rx] < self.rank[ry]:
            rx, ry = ry, rx
        self.parent[ry] = rx
        if self.rank[rx] == self.rank[ry]:
            self.rank[rx] += 1

cell_to_barcode_set = pairs_final_restricted.groupby('_match_key')['barcode'].apply(frozenset).to_dict()

all_cells_sc = pairs_final_restricted['_match_key'].unique().tolist()
cell_to_idx_sc = {c: i for i, c in enumerate(all_cells_sc)}
uf_sc = UnionFind(len(all_cells_sc))

n_merged = 0
n_checked = 0
for dom_clone, group in compare_df_r.groupby('dominant_clone'):
    members = group['cell'].tolist()
    if len(members) < 2:
        continue
    sets = {m: cell_to_barcode_set[m] for m in members if m in cell_to_barcode_set}
    members = list(sets.keys())
    for i in range(len(members)):
        for j in range(i + 1, len(members)):
            n_checked += 1
            si, sj = sets[members[i]], sets[members[j]]
            if si <= sj or sj <= si:
                uf_sc.union(cell_to_idx_sc[members[i]], cell_to_idx_sc[members[j]])
                n_merged += 1

print(f"\nchecked {n_checked} pairs within dominant clones, merged {n_merged} via subset-compatibility")

subset_components = {}
for cell, idx in cell_to_idx_sc.items():
    root = uf_sc.find(idx)
    subset_components.setdefault(root, []).append(cell)

clone_sizes_subset = pd.Series({root: len(m) for root, m in subset_components.items()})
print(f"\n[Subset-compatible] number of clones: {len(clone_sizes_subset)}")
print(f"[Subset-compatible] clone size distribution:\n{clone_sizes_subset.describe()}")

print(f"\n=== Comparison ===")
print(f"{'Dominant (loose)':30}{len(clone_sizes_dominant_r):>10}")
print(f"{'Subset-compatible':30}{len(clone_sizes_subset):>10}")
print(f"{'Exact-match (strict)':30}{len(clone_sizes_allbc_r):>10}")

In [ ]:
# ---- Look at the largest remaining splits post-filter ----

top_splits_r = split_dominant_r.sort_values(ascending=False).head(5)
for dom_clone, n_sub in top_splits_r.items():
    subset = compare_df_r[compare_df_r['dominant_clone'] == dom_clone]
    print(f"\ndominant clone '{dom_clone}' (size={len(subset)}) splits into {n_sub} exact-match groups:")
    sizes = subset.groupby('allbc_clone').size().sort_values(ascending=False)
    print(sizes.head(10))

    # show nUMI support for the secondary barcodes in a couple of the smaller sub-groups
    example_cells = subset[subset['allbc_clone'] != dom_clone]['cell'].head(5)
    print("\n  example split-off cells and their barcode support:")
    print(pairs_final_restricted[pairs_final_restricted['_match_key'].isin(example_cells)]
          [['_match_key', 'barcode', 'nUMI']].sort_values('_match_key'))

In [ ]:
"""
Preprocessing done to calculate PCA and also see if there is any sample effects (this is not finaly pre-proecessing)
Main Purpose:
1. Have PCA to compare cell similarity (to decide how to merge barcodes into clones)
2. Identify if there are any sample dependent read depth variation or any other batch effects - to decide if there is a need for sample-integration
Pipeline steps:
  1. Filter to singlets
  2. Inspect + filter on sequencing depth (min UMI count)
  3. Normalize (median-based total-count normalization)
  4. Log-transform
  5. HVG selection (batch-aware, so batch-driven genes aren't picked)
  6. Scale
  7. PCA
  8. Sanity check: does depth still leak into PC1?
  9. Neighbors + UMAP (uncorrected)
"""

import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
from scipy.stats import spearmanr


# ---- CONFIG ----
MIN_COUNTS = 500          # floor on total UMI counts per cell -- adjust based on your histogram
N_TOP_GENES = 2000         # number of HVGs
N_PCS = 50                 # number of PCs to compute
N_PCS_FOR_NEIGHBORS = 30   # number of PCs to use for neighbors/UMAP
BATCH_KEY = 'sample'       # column used for batch-aware HVG selection and integration


def main(loaded_data):
    adata = loaded_data.copy()

    # ---- 2. Depth QC ----
    adata.obs['total_counts'] = np.asarray(adata.X.sum(axis=1)).flatten()

    fig, ax = plt.subplots()
    ax.hist(adata.obs['total_counts'], bins=np.logspace(0, np.log10(adata.obs['total_counts'].max()), 60))
    ax.set_xscale('log')
    ax.axvline(MIN_COUNTS, color='k', linestyle='--', label=f'MIN_COUNTS = {MIN_COUNTS}')
    ax.set_xlabel('Total UMI counts per cell')
    ax.set_ylabel('Number of cells')
    ax.legend()
    plt.show()

    n_before = adata.n_obs
    sc.pp.filter_cells(adata, min_counts=MIN_COUNTS)
    print(f"depth filter (min_counts={MIN_COUNTS}): {n_before} -> {adata.n_obs} cells")

    print()
    print("depth per sample after filtering:")
    print(adata.obs.groupby('sample')['total_counts'].describe())

    # keep raw counts around for later (e.g. differential expression)
    adata.layers['counts'] = adata.X.copy()

    # ---- 3. Normalize (median-based) ----
    median_counts = np.median(adata.obs['total_counts'])
    print(f"\nnormalizing to median total counts: {median_counts}")
    sc.pp.normalize_total(adata, target_sum=None)  # None -> median

    # ---- 4. Log-transform ----
    sc.pp.log1p(adata)

    # ---- 5. HVG selection (batch-aware) ----
    sc.pp.highly_variable_genes(adata, n_top_genes=N_TOP_GENES, batch_key=BATCH_KEY)
    adata.raw = adata  # keep full gene set accessible via adata.raw
    adata = adata[:, adata.var['highly_variable']].copy()
    print(f"HVGs selected: {adata.n_vars}")

    # ---- 6. Scale ----
    sc.pp.scale(adata, max_value=10)

    # ---- 7. PCA ----
    sc.tl.pca(adata, n_comps=N_PCS, svd_solver='arpack')

    # ---- 8. Sanity check: depth vs PC1 ----
    rho, pval = spearmanr(adata.obsm['X_pca'][:, 0], adata.obs['total_counts'])
    print(f"\nSpearman correlation PC1 vs total_counts: rho={rho:.3f}, p={pval:.2e}")
    sc.pl.pca(adata, color='total_counts', title='PCA colored by depth')
    if abs(rho) > 0.3:
        print("WARNING: PC1 still correlates fairly strongly with sequencing depth. "
              "Consider a stricter MIN_COUNTS cutoff, or sc.pp.regress_out(adata, ['total_counts']) "
              "before scaling.")

    # ---- 9. Neighbors + UMAP (uncorrected) ----
    sc.pp.neighbors(adata, n_pcs=N_PCS_FOR_NEIGHBORS)
    sc.tl.umap(adata)
    sc.pl.umap(adata, color=['sample', 'folder_timepoint'], wspace=0.4,
               title=['Uncorrected UMAP - sample', 'Uncorrected UMAP - timepoint'])

    return adata


if __name__ == "__main__":
    # `loaded_data` must already be defined / loaded before running this script
    adata = main(combined)

In [ ]:
import numpy as np
import pandas as pd

# ===================================================================
# STEP 0: loaded_sample_names (skip if already defined)
# ===================================================================
loaded_sample_names = {
    'LSK_d2_1', 'LSK_d2_2', 'LSK_d2_3',
    'LSK_d4_1_1', 'LSK_d4_1_2', 'LSK_d4_1_3',
    'LSK_d4_2_1', 'LSK_d4_2_2', 'LSK_d4_2_3',
    'LSK_d6_1_1', 'LSK_d6_1_2', 'LSK_d6_1_3',
    'LSK_d6_2_1', 'LSK_d6_2_2', 'LSK_d6_2_3',
}

# ===================================================================
# STEP 1: Restrict pairs_final to loaded samples, build dominant + exact-match clones
# ===================================================================
pairs_final_restricted = pairs_final[pairs_final['sample'].isin(loaded_sample_names)].copy()
pairs_final_restricted['_match_key'] = pairs_final_restricted['sample'].astype(str) + "_" + pairs_final_restricted['cellID'].astype(str)

dominant_barcode_r = (
    pairs_final_restricted.sort_values('nUMI', ascending=False)
    .drop_duplicates('_match_key')
    .set_index('_match_key')['barcode']
)
clone_sizes_dominant_r = dominant_barcode_r.value_counts()

barcode_sets_r = pairs_final_restricted.groupby('_match_key')['barcode'].apply(lambda bcs: frozenset(bcs.unique()))
clone_sizes_allbc_r = barcode_sets_r.value_counts()
cell_to_allbc_clone_r = barcode_sets_r.apply(lambda s: ','.join(sorted(s)))

compare_df_r = pd.DataFrame({'cell': dominant_barcode_r.index, 'dominant_clone': dominant_barcode_r.values})
compare_df_r['allbc_clone'] = compare_df_r['cell'].map(cell_to_allbc_clone_r)

split_check_r = compare_df_r.groupby('dominant_clone')['allbc_clone'].nunique()
split_dominant_r = split_check_r[split_check_r > 1]
split_dominant_ids = set(split_dominant_r.index)

print(f"dominant clones: {len(clone_sizes_dominant_r)}, exact-match clones: {len(clone_sizes_allbc_r)}, "
      f"split dominant clones: {len(split_dominant_r)}")

cell_to_barcode_set = pairs_final_restricted.groupby('_match_key')['barcode'].apply(frozenset).to_dict()

disputed_df2 = compare_df_r[compare_df_r['dominant_clone'].isin(split_dominant_ids)].copy()
print(f"cells in disputed (split) dominant clones: {len(disputed_df2)}")

# ===================================================================
# STEP 2: PCA lookup from adata
# ===================================================================
N_PCS_FOR_SIM = 30
pca_lookup_key = adata.obs['sample'].astype(str) + "_" + pd.Series(adata.obs_names, index=adata.obs.index).astype(str)
pca_matrix = adata.obsm['X_pca'][:, :N_PCS_FOR_SIM]
key_to_pcvec = dict(zip(pca_lookup_key.values, pca_matrix))
key_to_sample = dict(zip(pca_lookup_key.values, combined.obs['sample'].astype(str).values))
all_keys = list(key_to_pcvec.keys())
print(f"cells with PCA embedding available: {len(key_to_pcvec)}")

def cosine_sim_matrix(vecs):
    norm = vecs / np.linalg.norm(vecs, axis=1, keepdims=True)
    return norm @ norm.T

# ===================================================================
# STEP 3: STRICT and LOOSE within-clone similarities (stratified same/cross-sample)
# ===================================================================
MAX_PAIRS_PER_CLONE = 2000
RNG = np.random.default_rng(0)

def within_group_similarities_stratified(clone_series, key_to_vec, key_to_samp, max_pairs_per_clone=MAX_PAIRS_PER_CLONE):
    same_sims, cross_sims = [], []
    for clone_id, members in clone_series.groupby(clone_series).groups.items():
        members = [m for m in members if m in key_to_vec]
        if len(members) < 2:
            continue
        vecs = np.array([key_to_vec[m] for m in members])
        samps = [key_to_samp[m] for m in members]
        sim_mat = cosine_sim_matrix(vecs)
        n = len(members)
        pairs_idx = list(zip(*np.triu_indices(n, k=1)))
        if len(pairs_idx) > max_pairs_per_clone:
            pairs_idx = [pairs_idx[i] for i in RNG.choice(len(pairs_idx), size=max_pairs_per_clone, replace=False)]
        for i, j in pairs_idx:
            s = sim_mat[i, j]
            (same_sims if samps[i] == samps[j] else cross_sims).append(s)
    return np.array(same_sims), np.array(cross_sims)

strict_key = barcode_sets_r.apply(lambda s: ','.join(sorted(s)))
strict_same, strict_cross = within_group_similarities_stratified(strict_key, key_to_pcvec, key_to_sample)
loose_same, loose_cross = within_group_similarities_stratified(dominant_barcode_r, key_to_pcvec, key_to_sample)
print(f"STRICT same={len(strict_same)}, cross={len(strict_cross)} | LOOSE same={len(loose_same)}, cross={len(loose_cross)}")

# ===================================================================
# STEP 4: Background similarity (fast vectorized)
# ===================================================================
keys_arr = np.array(all_keys)
vecs_arr = np.array([key_to_pcvec[k] for k in keys_arr])
samples_arr = np.array([key_to_sample[k] for k in keys_arr])
norm_vecs = vecs_arr / np.linalg.norm(vecs_arr, axis=1, keepdims=True)

N = len(keys_arr)
N_DRAWS = 2_000_000
rng2 = np.random.default_rng(1)
idx1 = rng2.integers(0, N, size=N_DRAWS)
idx2 = rng2.integers(0, N, size=N_DRAWS)
valid = idx1 != idx2
idx1, idx2 = idx1[valid], idx2[valid]
same_mask = samples_arr[idx1] == samples_arr[idx2]
sims_all = np.sum(norm_vecs[idx1] * norm_vecs[idx2], axis=1)
bg_same_pool = sims_all[same_mask]
bg_cross_pool = sims_all[~same_mask]

n_same_needed = len(loose_same)
n_cross_needed = len(loose_cross)
bg_same = bg_same_pool[:n_same_needed]
bg_cross = bg_cross_pool[:n_cross_needed]
print(f"BACKGROUND same={len(bg_same)} (mean={bg_same.mean():.4f}), cross={len(bg_cross)} (mean={bg_cross.mean():.4f})")

# ===================================================================
# STEP 5: Full 4-category breakdown
# ===================================================================
cat1_sims, cat1_sims_x = [], []
cat2_sims, cat2_sims_x = [], []
cat3_sims, cat3_sims_x = [], []
cat4_sims, cat4_sims_x = [], []

for dom_clone, group in disputed_df2.groupby('dominant_clone'):
    members = [m for m in group['cell'].tolist() if m in key_to_pcvec]
    if len(members) < 2:
        continue
    is_dominant_only = {m: (cell_to_barcode_set[m] == frozenset([dom_clone])) for m in members}

    vecs = np.array([key_to_pcvec[m] for m in members])
    samps = [key_to_sample[m] for m in members]
    norm = vecs / np.linalg.norm(vecs, axis=1, keepdims=True)
    sim_mat = norm @ norm.T
    n = len(members)

    for i, j in zip(*np.triu_indices(n, k=1)):
        s = sim_mat[i, j]
        cross = samps[i] != samps[j]
        mi, mj = members[i], members[j]
        set_i, set_j = cell_to_barcode_set[mi], cell_to_barcode_set[mj]
        both_dom_only = is_dominant_only[mi] and is_dominant_only[mj]
        one_dom_only = is_dominant_only[mi] != is_dominant_only[mj]

        if both_dom_only:
            (cat1_sims_x if cross else cat1_sims).append(s)
        elif set_i == set_j:
            (cat2_sims_x if cross else cat2_sims).append(s)
        elif one_dom_only:
            (cat3_sims_x if cross else cat3_sims).append(s)
        else:
            (cat4_sims_x if cross else cat4_sims).append(s)

cats = {
    "1. dominant-only vs dominant-only":       (np.array(cat1_sims), np.array(cat1_sims_x)),
    "2. dominant-plus vs dominant-plus (same)": (np.array(cat2_sims), np.array(cat2_sims_x)),
    "3. dominant-only vs dominant-plus":        (np.array(cat3_sims), np.array(cat3_sims_x)),
    "4. dominant-plus vs dominant-plus (diff)": (np.array(cat4_sims), np.array(cat4_sims_x)),
}

print("\n=== Full 4-category breakdown ===")
print(f"{'':45}{'n':>8}{'mean (same)':>14}{'n':>8}{'mean (cross)':>14}")
for label, (same, cross) in cats.items():
    same_mean = same.mean() if len(same) else float('nan')
    cross_mean = cross.mean() if len(cross) else float('nan')
    print(f"{label:45}{len(same):>8}{same_mean:>14.4f}{len(cross):>8}{cross_mean:>14.4f}")

print(f"\n(reference) BACKGROUND same-sample:  {bg_same.mean():.4f}")
print(f"(reference) BACKGROUND cross-sample: {bg_cross.mean():.4f}")

In [ ]:
# ---- Clone assignment via SUBSET-COMPATIBLE merging ----
# Two cells merge if one's barcode set is a subset of the other's. This is the rule
# validated by the similarity analysis: it merges categories 1/2/3 (all showed strong
# signal) but correctly leaves category 4 (different extras, weakest signal) unmerged.

class UnionFind:
    def __init__(self, n):
        self.parent = list(range(n))
        self.rank = [0] * n
    def find(self, x):
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]
            x = self.parent[x]
        return x
    def union(self, x, y):
        rx, ry = self.find(x), self.find(y)
        if rx == ry:
            return
        if self.rank[rx] < self.rank[ry]:
            rx, ry = ry, rx
        self.parent[ry] = rx
        if self.rank[rx] == self.rank[ry]:
            self.rank[rx] += 1

all_cells_sc = pairs_final_restricted['_match_key'].unique().tolist()
cell_to_idx_sc = {c: i for i, c in enumerate(all_cells_sc)}
uf_sc = UnionFind(len(all_cells_sc))

# Only compare cells within the same dominant clone -- cells outside it can't be
# subset-related anyway (they wouldn't share the dominant barcode at all).
n_merged = 0
n_checked = 0
for dom_clone, group in compare_df_r.groupby('dominant_clone'):
    members = group['cell'].tolist()
    if len(members) < 2:
        continue
    sets = {m: cell_to_barcode_set[m] for m in members if m in cell_to_barcode_set}
    members = list(sets.keys())
    for i in range(len(members)):
        for j in range(i + 1, len(members)):
            n_checked += 1
            si, sj = sets[members[i]], sets[members[j]]
            if si <= sj or sj <= si:  # subset check, either direction
                uf_sc.union(cell_to_idx_sc[members[i]], cell_to_idx_sc[members[j]])
                n_merged += 1

print(f"checked {n_checked} pairs within dominant clones, merged {n_merged} via subset-compatibility")

subset_components = {}
for cell, idx in cell_to_idx_sc.items():
    root = uf_sc.find(idx)
    subset_components.setdefault(root, []).append(cell)

clone_sizes_subset = pd.Series({root: len(m) for root, m in subset_components.items()})
print(f"\n[Subset-compatible] number of clones: {len(clone_sizes_subset)}")
print(f"[Subset-compatible] clone size distribution:\n{clone_sizes_subset.describe()}")

print(f"\n=== Comparison ===")
print(f"{'Dominant (loose)':30}{len(clone_sizes_dominant_r):>10}")
print(f"{'Subset-compatible':30}{len(clone_sizes_subset):>10}")
print(f"{'Exact-match (strict)':30}{len(clone_sizes_allbc_r):>10}")

Again back to using combined - adata was used for temporary pre-processing

In [ ]:
# ---- Match on cellID directly ----

# 1. Keep only label-sheet rows for samples we actually loaded
loaded_sample_names = set(combined.obs["sample"].unique())
cellLabelListSingleCell = cellLabelList[cellLabelList['sample'].isin(loaded_sample_names)].copy()

# 2. Use cellID (hyphen-stripped, in case it has the same '-' formatting as before) as the match key
cellLabelListSingleCell['_match_key'] = (
    cellLabelListSingleCell['sample'].astype(str) + "_" + cellLabelListSingleCell['cellID']
)
dup_labels = cellLabelListSingleCell['_match_key'].duplicated().sum()
if dup_labels:
    print(f"WARNING: {dup_labels} duplicate cellID keys in label sheet -- keeping first")
cellLabelListSingleCell = cellLabelListSingleCell.drop_duplicates(subset='_match_key').reset_index(drop=True)

# 3. Build the label lookup
cellIDLabels = cellLabelListSingleCell.drop(columns=['cellID', 'barcode', 'sample', 'nUMI'])
labelID = cellIDLabels.set_index(cellLabelListSingleCell['_match_key'])

# 4. Sanity check overlap BEFORE mapping, against combined's existing sample+barcode match key
overlap = pd.Index(cellLabelListSingleCell['_match_key']).intersection(combined.obs['_match_key'])
print(f"overlap between cellID key and combined's sample_barcode key: {len(overlap)} / {combined.n_obs}")

print()
print("cellID match_key samples:", cellLabelListSingleCell['_match_key'].head(5).tolist())
print("combined match_key samples:", combined.obs['_match_key'].head(5).tolist())

# 5. Only assign if the overlap actually looks reasonable
combined.obs["singletStatus"] = combined.obs["_match_key"].map(labelID["label"])
print()
print(combined.obs["singletStatus"].value_counts(dropna=False))

In [ ]:
# ---- Assign clone_id to combined.obs, using the subset-compatible clone definition ----

# 1. Sanity check: every subset-component should share one common dominant_clone
#    (since the union step only ever compared cells within the same dominant clone group)
cell_to_dominant = compare_df_r.set_index('cell')['dominant_clone'].to_dict()

inconsistent = 0
for root, members in subset_components.items():
    doms = {cell_to_dominant.get(m) for m in members if m in cell_to_dominant}
    if len(doms) > 1:
        inconsistent += 1
print(f"subset-components spanning >1 dominant clone (should be 0): {inconsistent}")

# 2. Build a readable, stable clone_id: <dominant_barcode>__sub<k>
#    (k = 1, 2, 3... when a dominant clone was split into multiple subset-components,
#    ordered largest-to-smallest for stability)
clone_id_map = {}
by_dominant = {}
for root, members in subset_components.items():
    dom = next((cell_to_dominant.get(m) for m in members if m in cell_to_dominant), None)
    by_dominant.setdefault(dom, []).append((root, members))

for dom, comps in by_dominant.items():
    comps_sorted = sorted(comps, key=lambda x: -len(x[1]))
    for k, (root, members) in enumerate(comps_sorted, start=1):
        suffix = "" if len(comps_sorted) == 1 else f"__sub{k}"
        clone_id = f"{dom}{suffix}"
        for m in members:
            clone_id_map[m] = clone_id

print(f"total cells assigned a clone_id: {len(clone_id_map)}")
print(f"total distinct clone_ids: {len(set(clone_id_map.values()))}")

# 3. Map onto combined.obs
combined.obs["clone_id"] = combined.obs["_match_key"].map(clone_id_map)

print()
print(f"cells with a clone_id assigned: {combined.obs['clone_id'].notna().sum()} / {combined.n_obs}")
print()
print("clone size distribution (in combined.obs):")
print(combined.obs["clone_id"].value_counts().describe())

In [ ]:
combined = combined[combined.obs['singletStatus'] == "Singlet"].copy()

In [ ]:
# ---- Save combined to h5ad ----

# h5ad requires obs columns to be simple dtypes (str/category/numeric) -- 'object' columns
# with mixed types (e.g. strings mixed with NaN, or Python None) can throw write errors.
# Clean these up first.

import pandas as pd

for col in ["sample", ]:
    if combined.obs[col].dtype == object:
        # convert to string, keeping NaN as NaN (not the string "nan")
        combined.obs[col] = combined.obs[col].astype("string").astype("object")

# drop any leftover internal helper columns you don't want saved (optional)
helper_cols = [c for c in combined.obs.columns if c.startswith('_')]
if helper_cols:
    print(f"dropping helper columns before saving: {helper_cols}")
    combined.obs = combined.obs.drop(columns=helper_cols)

OUTPUT_PATH = "combined_with_clones_filtered.h5ad"
combined.write_h5ad(OUTPUT_PATH)
print(f"saved to {OUTPUT_PATH}")

# quick sanity check: read it back
import scanpy as sc
check = sc.read_h5ad(OUTPUT_PATH)
print(check)

In [ ]:
# ---- Step 0: check the h5ad's actual ground-truth label format first ----
print("h5ad ground-truth cell type annotation values:")
print(h5ad.obs['Cell type annotation'].value_counts(dropna=False))

In [ ]:
%pip install leidenalg


In [ ]:
# ===================================================================
# STEP G: Compare each method to ground truth (fixed for duplicate obs_names)
# ===================================================================
valid = adata.obs['ground_truth'].notna()

pred1 = adata.obs.loc[valid, 'cell_type_cluster_harm'].values
pred3 = adata.obs.loc[valid, 'cell_type_manual_harm'].values
truth = adata.obs.loc[valid, 'ground_truth'].values

print("\n=== Method 1 (cluster-based) vs ground truth ===")
conf1 = pd.crosstab(pred1, truth, rownames=['Predicted'], colnames=['Ground truth'])
print(conf1)
acc1 = (pred1 == truth).mean()
print(f"\noverall agreement (Method 1): {acc1:.4f}")

print("\n=== Method 3 (per-cell) vs ground truth ===")
conf3 = pd.crosstab(pred3, truth, rownames=['Predicted'], colnames=['Ground truth'])
print(conf3)
acc3 = (pred3 == truth).mean()
print(f"\noverall agreement (Method 3): {acc3:.4f}")

In [ ]:
# ===================================================================
# STEP H: Combined method -- per-cell score + per-cell graph-neighborhood agreement
# ("well separated from alternative lineage branches", applied per-cell rather
# than per-cluster, using the same idea as the earlier neighbor-purity check)
# ===================================================================
import numpy as np
from scipy.sparse import find

PURITY_THRESHOLD = 0.3  # fraction of a cell's neighbors that must share its call to keep it

connectivities = adata.obsp['connectivities']
labels = adata.obs['cell_type_manual'].astype(str).values  # Method 3's raw per-cell calls
n = len(labels)

rows, cols, _ = find(connectivities)
same_label = (labels[rows] == labels[cols]).astype(float)

agreement_num = np.bincount(rows, weights=same_label, minlength=n)
agreement_denom = np.bincount(rows, minlength=n)
agreement_frac = np.divide(agreement_num, agreement_denom,
                            out=np.zeros(n), where=agreement_denom > 0)

adata.obs['neighbor_agreement'] = agreement_frac

# Only trust the per-cell call if it's also echoed by most of the cell's neighbors;
# otherwise fall back to 'unassigned' (i.e. treat as undifferentiated/ambiguous)
adata.obs['cell_type_combined'] = np.where(
    agreement_frac >= PURITY_THRESHOLD,
    adata.obs['cell_type_manual'].astype(str),
    'unassigned'
)

print("Combined method label counts:")
print(adata.obs['cell_type_combined'].value_counts())

print(f"\ncells changed from a real call to 'unassigned' by the purity filter: "
      f"{((adata.obs['cell_type_manual'] != 'unassigned') & (adata.obs['cell_type_combined'] == 'unassigned')).sum()}")

# ===================================================================
# STEP I: Compare combined method to ground truth
# ===================================================================
adata.obs['cell_type_combined_harm'] = adata.obs['cell_type_combined'].map(label_harmonize)

valid = adata.obs['ground_truth'].notna()
pred_combined = adata.obs.loc[valid, 'cell_type_combined_harm'].values
truth = adata.obs.loc[valid, 'ground_truth'].values

print("\n=== Combined method vs ground truth ===")
conf_combined = pd.crosstab(pred_combined, truth, rownames=['Predicted'], colnames=['Ground truth'])
print(conf_combined)
acc_combined = (pred_combined == truth).mean()
print(f"\noverall agreement (Combined): {acc_combined:.4f}")

print(f"\n=== Summary across all three methods ===")
print(f"Method 1 (cluster):     {acc1:.4f}")
print(f"Method 3 (per-cell):    {acc3:.4f}")
print(f"Combined (per-cell + neighbor agreement): {acc_combined:.4f}")

In [ ]:
# ===================================================================
# STEP J: Semi-supervised classification -- train on a subset of ground truth
# ===================================================================
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
import pandas as pd
import numpy as np

# 1. Restrict to cells that actually have ground truth
has_gt = adata.obs['ground_truth'].notna()
gt_idx = np.where(has_gt.values)[0]
gt_labels = adata.obs['ground_truth'].values[gt_idx]

print("class counts among ground-truth cells:")
print(pd.Series(gt_labels).value_counts())

# 2. Stratified train/test split -- e.g. 80/20, so rare classes appear in both
TRAIN_FRACTION = 0.8
train_idx, test_idx = train_test_split(
    gt_idx, train_size=TRAIN_FRACTION, stratify=gt_labels, random_state=0
)
print(f"\ntrain: {len(train_idx)} cells, test: {len(test_idx)} cells")

# 3. Features: PCA embedding (already computed on adata)
X_train = adata.obsm['X_pca'][train_idx]
y_train = adata.obs['ground_truth'].values[train_idx]
X_test = adata.obsm['X_pca'][test_idx]
y_test = adata.obs['ground_truth'].values[test_idx]

# 4. Train a classifier with balanced class weighting, so rare types aren't ignored
clf = RandomForestClassifier(
    n_estimators=300, class_weight='balanced_subsample', random_state=0, n_jobs=-1
)
clf.fit(X_train, y_train)

# 5. Evaluate on the held-out test split
y_pred_test = clf.predict(X_test)
print("\n=== Classifier performance on held-out test set ===")
print(classification_report(y_test, y_pred_test))

conf = pd.crosstab(pd.Series(y_pred_test, name='Predicted'), pd.Series(y_test, name='Ground truth'))
print(conf)

# 6. Predict on ALL cells (including those with no ground truth at all)
X_all = adata.obsm['X_pca']
adata.obs['cell_type_classifier'] = clf.predict(X_all)

print("\nfull-dataset predicted label counts:")
print(adata.obs['cell_type_classifier'].value_counts())

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import classification_report, accuracy_score

# ===================================================================
# STEP 1: Marker gene sets (Table 2) + per-cell scoring
# ===================================================================
marker_sets = {
    'Mast cell (Ma)':          ['Cma1', 'Tph1', 'Papss2', 'Fcer1a', 'Gzmb'],
    'Basophil (Ba)':           ['Hgf', 'Ccl3', 'Slpi'],
    'Eosinophil (Eos)':        ['Prg3', 'Epx', 'Prg2'],
    'Megakaryocyte (Mk)':      ['Ppbp', 'Thbs1', 'Pf4', 'Timp3'],
    'Monocyte (Mo)':           ['Ms4a6d', 'Fabp5', 'Ctss', 'Ms4a6c', 'Tgfbi', 'Olfm1',
                                 'Csf1r', 'Ccr2', 'Klf4', 'F13a1'],
    'Neutrophil (Neu)':        ['S100a9', 'Itgb2l', 'Elane', 'Fcnb', 'Mpo', 'Prtn3',
                                 'S100a6', 'S100a8', 'Lcn2', 'Lrg1'],
    'Lymphoid (Ly)':           ['Ighm', 'Satb1', 'Dntt', 'Ctr9', 'Jchain'],
    'Migratory DC (migDC)':    ['H2-Eb1', 'H2-Aa', 'H2-Ab1', 'H2-DMb1', 'Ccr7'],
    'Plasmacytoid DC (pDC)':   ['Siglech'],
    'Erythroid (Er)':          ['Hbb-bs'],
    'Classical DC (cDC)':      ['Cst3', 'Xcr1'],
    'B cell (B)':              ['Vpreb1', 'Vpreb2', 'Cd79a'],
    'T cell (T)':              ['Bcl11b', 'Trdc', 'Lck'],
    'NK cell (NK)':            ['Gzma'],
}

score_source_var_names = adata.raw.var_names if adata.raw is not None else adata.var_names

score_cols = []
for cell_type, genes in marker_sets.items():
    genes_present = [g for g in genes if g in score_source_var_names]
    if not genes_present:
        print(f"WARNING: no genes found for {cell_type} -- skipping")
        continue
    score_name = f"score_{cell_type.split(' (')[0].replace(' ', '_')}"
    sc.tl.score_genes(adata, genes_present, score_name=score_name, use_raw=(adata.raw is not None))
    score_cols.append(score_name)

# ===================================================================
# STEP 2: Match ground truth from h5ad
# ===================================================================
adata.obs['_match_key'] = adata.obs['sample'].astype(str) + "_" + pd.Series(
    [strip_suffix(b) for b in adata.obs_names], index=adata.obs.index
)
h5ad.obs['_match_key'] = h5ad.obs[H5AD_LIBRARY_COL].astype(str) + "_" + pd.Series(
    [strip_suffix(b) for b in h5ad.obs_names], index=h5ad.obs.index
)
gt_lookup = h5ad.obs.set_index('_match_key')['Cell type annotation']
adata.obs['ground_truth'] = adata.obs['_match_key'].map(gt_lookup)
print(f"cells with ground truth: {adata.obs['ground_truth'].notna().sum()} / {adata.n_obs}")

# ===================================================================
# STEP 3: Build combined features (PCA + marker scores) for ALL cells
# ===================================================================
N_PCS_FOR_CLF = 30
X_pca_part = adata.obsm['X_pca'][:, :N_PCS_FOR_CLF]
X_scores_part = adata.obs[score_cols].values
X_combined = np.hstack([X_pca_part, X_scores_part])
print(f"feature matrix shape: {X_combined.shape}")

has_gt = adata.obs['ground_truth'].notna()
gt_idx = np.where(has_gt.values)[0]
X_gt = X_combined[gt_idx]
y_gt = adata.obs['ground_truth'].values[gt_idx]

# ===================================================================
# STEP 4: Cross-validated evaluation (k-NN base classifier)
# ===================================================================
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
knn = KNeighborsClassifier(n_neighbors=15, weights='distance')

y_proba_cv = cross_val_predict(knn, X_gt, y_gt, cv=cv, method='predict_proba', n_jobs=-1)
knn_temp = knn.fit(X_gt, y_gt)  # just to get .classes_ ordering
proba_df = pd.DataFrame(y_proba_cv, columns=knn_temp.classes_)

# ---- Erythroid probability-threshold override (base k-NN argmax fails on this class) ----
ERYTHROID_THRESHOLD = 0.10  # chosen from the earlier precision/recall sweep

base_pred_cv = proba_df.idxmax(axis=1).values
final_pred_cv = np.where(
    proba_df['Erythroid'].values >= ERYTHROID_THRESHOLD,
    'Erythroid',
    base_pred_cv
)

print("\n=== Cross-validated performance of FINAL combined method ===")
print(classification_report(y_gt, final_pred_cv))
print(f"overall accuracy: {accuracy_score(y_gt, final_pred_cv):.4f}")

# ===================================================================
# STEP 5: Train final model on ALL ground truth, predict on EVERY cell
# ===================================================================
knn_final = KNeighborsClassifier(n_neighbors=15, weights='distance').fit(X_gt, y_gt)
proba_all = knn_final.predict_proba(X_combined)
proba_all_df = pd.DataFrame(proba_all, columns=knn_final.classes_, index=adata.obs.index)

base_pred_all = knn_final.predict(X_combined)
final_pred_all = np.where(
    proba_all_df['Erythroid'].values >= ERYTHROID_THRESHOLD,
    'Erythroid',
    base_pred_all
)

adata.obs['cell_type_final'] = final_pred_all

print(f"\ntotal cells: {adata.n_obs}")
print(f"cells with real ground truth: {has_gt.sum()}")
print(f"cells newly classified by the model: {(~has_gt).sum()}")
print("\nfinal label counts:")
print(adata.obs['cell_type_final'].value_counts())

In [ ]:
# ---- Venn-style overlap between the label sheet and combined ----

A = set(cellLabelListSingleCell['_match_key'])   # cells in the label sheet (restricted to loaded samples)
B = set(combined.obs['_match_key'])            # cells in combined

only_in_A = A - B
only_in_B = B - A
in_both = A & B

print(f"A (label sheet, loaded samples only): {len(A)}")
print(f"B (combined):                      {len(B)}")
print(f"A \\ B (sheet only):                   {len(only_in_A)}")
print(f"B \\ A (combined only):             {len(only_in_B)}")
print(f"A ∩ B (in both):                      {len(in_both)}")
print(f"A ∪ B (union):                        {len(A | B)}")

# ---- Venn diagram with singlet/multiplet breakdown of the overlap ----

from matplotlib_venn import venn2
import matplotlib.pyplot as plt

overlap_labels = cellLabelListSingleCell.set_index('_match_key').loc[list(in_both), 'label']
status_counts = overlap_labels.value_counts(dropna=False)

fig, ax = plt.subplots(figsize=(7, 6))
venn2(subsets=(len(only_in_A), len(only_in_B), len(in_both)),
      set_labels=('Label sheet', 'LARRY data'), ax=ax)
ax.set_title("Cell overlap: label sheet vs combined")

breakdown_text = "\n".join(f"{status}: {count}" for status, count in status_counts.items())
ax.text(0.5, -0.15, f"Within overlap (A ∩ B):\n{breakdown_text}",
        transform=ax.transAxes, ha='center', va='top', fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
# ---- Sequencing depth (total UMI counts) per sample ----
import numpy as np

combined.obs['total_counts'] = np.asarray(combined.X.sum(axis=1)).flatten()

depth_summary = combined.obs.groupby('sample')['total_counts'].describe()
print(depth_summary)

# quick visual comparison across samples
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 5))
combined.obs.boxplot(column='total_counts', by='sample', ax=ax, rot=90)
ax.set_yscale('log')
ax.set_ylabel('Total UMI counts per cell (log scale)')
plt.title('Per-cell sequencing depth by sample')
plt.suptitle('')
plt.tight_layout()
plt.show()

In [ ]:
# ---- What's driving PC1 and PC2? ----

sc.pl.pca(adata, color=['sample', 'folder_timepoint'], wspace=0.4,
           title=['PCA - sample', 'PCA - timepoint'])

# top genes loading onto PC1 and PC2
sc.pl.pca_loadings(adata, components=[1, 2])

In [ ]:
# ---- Mitochondrial gene percentage (correct: run on RAW counts, full gene set) ----
qc_check = combined.copy()  # raw counts, all genes -- NOT the HVG-subsetted/scaled `adata`
qc_check.var['mt'] = qc_check.var_names.str.startswith('mt-')
print(f"mitochondrial genes found: {qc_check.var['mt'].sum()}")
print(qc_check.var_names[qc_check.var['mt']].tolist())
sc.pp.calculate_qc_metrics(qc_check, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)
print(qc_check.obs['pct_counts_mt'].describe())
sc.pl.violin(qc_check, ['pct_counts_mt'], groupby='sample', rotation=90)
# carry the result back onto combined / adata for downstream filtering if needed
combined.obs['pct_counts_mt'] = qc_check.obs['pct_counts_mt'].values